In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

from pyspark_data_provenance import (
    data_provenance_enabled, 
    build_data_provenance_session
)

In [3]:
from pyspark.sql import SparkSession


spark = (
    # SparkSession
    # .builder
    build_data_provenance_session()
    .appName("data-provenance-notebook")
    .getOrCreate()
)


26/04/22 10:40:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

false
true
false


## Create toy dataframes

In [5]:
from datetime import date

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("C", date(2026, 1, 17), 80.0, 60),
    ("F", date(2026, 1, 16), 50.0, 70)
], ["product", "date", "price", "quantity"]
)
df.printSchema()

df.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)



,product,date,price,quantity
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,C,2026-01-17,80.0,60
6,F,2026-01-16,50.0,70


In [25]:
df2 = spark.createDataFrame([
    ("A", "Bike"),
    ("B", "Handball"),
    ("C", "Bike"),
    ("D", "Handball"),
    ("E", "Running")
],["letter","labell"]
)

df2.printSchema()
df2.toPandas()



root
 |-- letter: string (nullable = true)
 |-- labell: string (nullable = true)



,letter,labell
0,A,Bike
1,B,Handball
2,C,Bike
3,D,Handball
4,E,Running


## Tests with data_provenance_enabled

### Select

In [28]:
with data_provenance_enabled(spark):
    df3 = df.select("product")
df3.toPandas()

,product,_provenance_tag
0,A,8589934592
1,A,17179869184
2,A,34359738368
3,B,42949672960
4,B,60129542144
5,C,68719476736
6,F,77309411328


In [36]:
with data_provenance_enabled(spark):
    df4 = df2.select("letter")
df4.toPandas()

,letter,_provenance_tag
0,A,8589934592
1,B,25769803776
2,C,42949672960
3,D,60129542144
4,E,77309411328


In [ ]:
df.createOrReplaceTempView("sales")
with data_provenance_enabled(spark):
    res = spark.sql("select product from sales")
res.toPandas()

,product,_provenance_tag
0,A,8589934592
1,A,17179869184
2,A,34359738368
3,B,42949672960
4,B,60129542144
5,C,68719476736
6,F,77309411328


In [ ]:
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark):
    res2 = spark.sql("select letter from category")
res2.toPandas()

,letter,_provenance_tag
0,A,8589934592
1,B,25769803776
2,C,42949672960
3,D,60129542144
4,E,77309411328


### Join

In [40]:
df.createOrReplaceTempView("sales")
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark):
    res = spark.sql("select * from sales s join category c on s.product=c.letter").toPandas()

res

,product,date,price,quantity,letter,labell,_provenance_tag
0,A,2026-01-15,10.0,90,A,Bike,0
1,A,2026-01-16,10.0,120,A,Bike,1
2,A,2026-01-17,5.0,300,A,Bike,2
3,B,2026-01-15,100.0,20,B,Handball,3
4,B,2026-01-16,100.0,30,B,Handball,4
5,C,2026-01-17,80.0,60,C,Bike,5


In [44]:

with data_provenance_enabled(spark):
    df5 = df.select("product").join(df2, df.product == df2.letter)

df5.toPandas()

,product,_provenance_tag,letter,labell
0,A,8589934592,A,Bike
1,A,17179869184,A,Bike
2,A,34359738368,A,Bike
3,B,42949672960,B,Handball
4,B,60129542144,B,Handball
5,C,68719476736,C,Bike


In [60]:

with data_provenance_enabled(spark):
    df6 = df.join(df2, df.product == df2.letter).select("product")

df6.toPandas()

,product,_provenance_tag
0,A,0
1,A,1
2,A,2
3,B,3
4,B,4
5,C,5


### Where

In [11]:
df3 = df.select("*").filter("price > 10")
df3.toPandas()
#df3.explain("extended")

,product,date,price,quantity
0,B,2026-01-15,100.0,20
1,B,2026-01-16,100.0,30
2,C,2026-01-17,80.0,60
3,F,2026-01-16,50.0,70


In [ ]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select product, price from sales where price>10").toPandas()

res

,product,price,_provenance_tag
0,B,100.0,42949672960
1,B,100.0,60129542144
2,C,80.0,68719476736
3,F,50.0,77309411328


### Other tests

In [58]:
df4 = df.select("*").filter("price > 10").sort("price").join(df2, df.product == df2.letter)
df4.toPandas()
#df4.explain(True)

,product,date,price,quantity,letter,labell
0,B,2026-01-15,100.0,20,B,Handball
1,B,2026-01-16,100.0,30,B,Handball
2,C,2026-01-17,80.0,60,C,Bike


In [14]:
df6 = df.select("*").join(df2, "product", "outer")
df6.toPandas()

,product,date,price,quantity,labell
0,A,2026-01-15,10.0,90.0,Bike
1,A,2026-01-16,10.0,120.0,Bike
2,A,2026-01-17,5.0,300.0,Bike
3,B,2026-01-15,100.0,20.0,Handball
4,B,2026-01-16,100.0,30.0,Handball
5,C,2026-01-17,80.0,60.0,Bike
6,D,None,NaN,NaN,Handball
7,E,None,NaN,NaN,Running
8,F,2026-01-16,50.0,70.0,NaN


In [15]:
with data_provenance_enabled(spark) :
    dfy = df.select("*").join(df2, "product", "outer")
    dfy.toPandas()

In [55]:
df7 = df.groupBy("product").agg({"quantity": "sum"}).sort("product")
df7.toPandas()

,product,sum(quantity)
0,A,510
1,B,50
2,C,60
3,F,70


In [54]:
df8 = df.withColumn("revenue", df["quantity"] * df["price"]) \
        .groupBy("product") \
        .agg({"revenue": "sum"}) \
        .sort("product")

df8.toPandas()
df8.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 2
   +- *(3) Sort [product#0 ASC NULLS FIRST], true, 0
      +- AQEShuffleRead coalesced
         +- ShuffleQueryStage 1
            +- Exchange rangepartitioning(product#0 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=1529]
               +- *(2) HashAggregate(keys=[product#0], functions=[sum(revenue#52)])
                  +- AQEShuffleRead coalesced
                     +- ShuffleQueryStage 0
                        +- Exchange hashpartitioning(product#0, 200), ENSURE_REQUIREMENTS, [plan_id=1505]
                           +- *(1) HashAggregate(keys=[product#0], functions=[partial_sum(revenue#52)])
                              +- *(1) Project [product#0, (cast(quantity#3L as double) * price#2) AS revenue#52]
                                 +- *(1) Scan ExistingRDD[product#0,date#1,price#2,quantity#3L]
+- == Initial Plan ==
   Sort [product#0 ASC NULLS FIRST], true, 0
   +- Exchang